# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.004831,0.002142,0.993027,0.005220,0.002128,0.992652,0.005655,0.000463,0.993882
1,0.998229,0.000143,0.001628,0.997774,0.000322,0.001904,0.994638,0.000368,0.004994
2,0.003982,0.000584,0.995434,0.002362,0.000710,0.996928,0.002327,0.000919,0.996754
3,0.007424,0.001954,0.990622,0.002922,0.003628,0.993450,0.003433,0.002374,0.994194
4,0.993583,0.000012,0.006405,0.997151,0.000013,0.002836,0.994112,0.000005,0.005883


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.011986,0.002516,0.985498,0.009749,0.003956,0.986295,0.008185,0.005526,0.986288
1,0.469590,0.002922,0.527488,0.344531,0.002104,0.653365,0.530403,0.002094,0.467503
2,0.998492,0.000991,0.000517,0.998555,0.001132,0.000313,0.996306,0.003216,0.000478
3,0.986619,0.000009,0.013372,0.978274,0.000024,0.021702,0.990683,0.000093,0.009224
4,0.008520,0.001975,0.989505,0.005601,0.002080,0.992319,0.003507,0.001947,0.994546


In [6]:
y_train.head()

,health_condition
id,
0,2
1,0
2,2
3,2
4,0


# Machine Learning

In [7]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [29]:
def objective(trial, X, y):

    lgbm_w0 = trial.suggest_float('weight_lgbm_class_0', 0.0, 1.0)
    lgbm_w1 = trial.suggest_float('weight_lgbm_class_1', 0.0, 1.0)
    lgbm_w2 = trial.suggest_float('weight_lgbm_class_2', 0.0, 1.0)
    
    xgb_w0 = trial.suggest_float('weight_xgb_class_0', 0.0, 1.0)
    xgb_w1 = trial.suggest_float('weight_xgb_class_1', 0.0, 1.0)
    xgb_w2 = trial.suggest_float('weight_xgb_class_2', 0.0, 1.0)
    
    cat_w0 = trial.suggest_float('weight_cat_class_0', 0.0, 1.0)
    cat_w1 = trial.suggest_float('weight_cat_class_1', 0.0, 1.0)
    cat_w2 = trial.suggest_float('weight_cat_class_2', 0.0, 1.0)
    
    proba_lgbm = X[['lgbm_0', 'lgbm_1', 'lgbm_2']].to_numpy()
    proba_xgb  = X[['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
    proba_cat  = X[['cat_0', 'cat_1', 'cat_2']].to_numpy()

    w_lgbm = np.array([lgbm_w0, lgbm_w1, lgbm_w2])
    w_xgb  = np.array([xgb_w0, xgb_w1, xgb_w2])
    w_cat  = np.array([cat_w0, cat_w1, cat_w2])
    
    blend_proba = (proba_lgbm * w_lgbm) + (proba_xgb * w_xgb) + (proba_cat * w_cat)
    
    pred = np.argmax(blend_proba, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=2000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-03 10:22:35,846] A new study created in memory with name: no-name-e3ddce24-50c4-4669-9d42-38fab62f9171
Best trial: 5. Best value: 0.933954:   0%|▋                                                                                                                                      | 10/2000 [00:00<01:12, 27.53it/s]

[I 2026-07-03 10:22:36,024] Trial 1 finished with value: 0.8689888641616194 and parameters: {'weight_lgbm_class_0': 0.963272853179441, 'weight_lgbm_class_1': 0.913928441531453, 'weight_lgbm_class_2': 0.19765012096309487, 'weight_xgb_class_0': 0.30442554182296644, 'weight_xgb_class_1': 0.8589030048666338, 'weight_xgb_class_2': 0.9193728544684141, 'weight_cat_class_0': 0.8912555837284181, 'weight_cat_class_1': 0.5577475158449899, 'weight_cat_class_2': 0.045545943321797955}. Best is trial 1 with value: 0.8689888641616194.
[I 2026-07-03 10:22:36,033] Trial 2 finished with value: 0.8609455595018574 and parameters: {'weight_lgbm_class_0': 0.8055997008613686, 'weight_lgbm_class_1': 0.1377182403851992, 'weight_lgbm_class_2': 0.484715367941861, 'weight_xgb_class_0': 0.10574691282929005, 'weight_xgb_class_1': 0.42834695126475353, 'weight_xgb_class_2': 0.6293545638425081, 'weight_cat_class_0': 0.43135243124079115, 'weight_cat_class_1': 0.08858783189058794, 'weight_cat_class_2': 0.0996034997819340

Best trial: 19. Best value: 0.948925:   1%|█▏                                                                                                                                    | 18/2000 [00:00<00:58, 34.09it/s]

[I 2026-07-03 10:22:36,149] Trial 11 finished with value: 0.898278591904759 and parameters: {'weight_lgbm_class_0': 0.6526514680791069, 'weight_lgbm_class_1': 0.7125104951727536, 'weight_lgbm_class_2': 0.9374387509179225, 'weight_xgb_class_0': 0.34237101817575444, 'weight_xgb_class_1': 0.4303406746538798, 'weight_xgb_class_2': 0.4060951500742087, 'weight_cat_class_0': 0.27224998332915284, 'weight_cat_class_1': 0.5415014051643281, 'weight_cat_class_2': 0.4244055553623516}. Best is trial 5 with value: 0.933954113621815.
[I 2026-07-03 10:22:36,230] Trial 14 finished with value: 0.8457708044541018 and parameters: {'weight_lgbm_class_0': 0.874082165266185, 'weight_lgbm_class_1': 0.581196955398047, 'weight_lgbm_class_2': 0.39135913796513055, 'weight_xgb_class_0': 0.9161096053404489, 'weight_xgb_class_1': 0.06296622510310745, 'weight_xgb_class_2': 0.5534996173588128, 'weight_cat_class_0': 0.9617765921923006, 'weight_cat_class_1': 0.4285588904175286, 'weight_cat_class_2': 0.39071541675119137}.

Best trial: 19. Best value: 0.948925:   1%|█▋                                                                                                                                    | 26/2000 [00:00<00:54, 36.48it/s]

...

[I 2026-07-03 10:24:01,694] Trial 1987 finished with value: 0.9497049039005807 and parameters: {'weight_lgbm_class_0': 0.05158189861476428, 'weight_lgbm_class_1': 0.7224883841528768, 'weight_lgbm_class_2': 0.8976123214801294, 'weight_xgb_class_0': 0.03942335560901475, 'weight_xgb_class_1': 0.4099662701129907, 'weight_xgb_class_2': 0.5902266489287339, 'weight_cat_class_0': 0.03150015337539688, 'weight_cat_class_1': 0.7663494362087383, 'weight_cat_class_2': 0.07846674274127997}. Best is trial 1765 with value: 0.9497917773888238.
[I 2026-07-03 10:24:01,834] Trial 1989 finished with value: 0.9497286238068053 and parameters: {'weight_lgbm_class_0': 0.0522593295028281, 'weight_lgbm_class_1': 0.7187155600676031, 'weight_lgbm_class_2': 0.9010387443831286, 'weight_xgb_class_0': 0.03479939075657576, 'weight_xgb_class_1': 0.4552341652454819, 'weight_xgb_class_2': 0.5911465638773743, 'weight_cat_class_0': 0.03401826067864696, 'weight_cat_class_1': 0.6648944848927323, 'weight_cat_class_2': 0.083292

Best trial: 1765. Best value: 0.949792: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 1997/2000 [01:26<00:00, 16.48it/s]

[I 2026-07-03 10:24:01,960] Trial 1991 finished with value: 0.9497101647372191 and parameters: {'weight_lgbm_class_0': 0.04689999110244088, 'weight_lgbm_class_1': 0.5891656211724923, 'weight_lgbm_class_2': 0.8516680893876082, 'weight_xgb_class_0': 0.03275408568076724, 'weight_xgb_class_1': 0.41740024655859675, 'weight_xgb_class_2': 0.45156049235982454, 'weight_cat_class_0': 0.032182936093434485, 'weight_cat_class_1': 0.9671913713271669, 'weight_cat_class_2': 0.09742613383060438}. Best is trial 1765 with value: 0.9497917773888238.
[I 2026-07-03 10:24:02,011] Trial 1992 finished with value: 0.9496722965931849 and parameters: {'weight_lgbm_class_0': 0.05108275308326486, 'weight_lgbm_class_1': 0.7223669571092621, 'weight_lgbm_class_2': 0.8567170059299749, 'weight_xgb_class_0': 0.0327434240936472, 'weight_xgb_class_1': 0.5167210962834698, 'weight_xgb_class_2': 0.5973670466875604, 'weight_cat_class_0': 0.03418214983095687, 'weight_cat_class_1': 0.6844723575011473, 'weight_cat_class_2': 0.226

Best trial: 1765. Best value: 0.949792: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [01:26<00:00, 23.16it/s]

[I 2026-07-03 10:24:02,187] Trial 1999 finished with value: 0.9497460205839158 and parameters: {'weight_lgbm_class_0': 0.01749520504140869, 'weight_lgbm_class_1': 0.6709760762759878, 'weight_lgbm_class_2': 0.8558541251621312, 'weight_xgb_class_0': 0.030866146882310794, 'weight_xgb_class_1': 0.5417621275010274, 'weight_xgb_class_2': 0.4139053899480577, 'weight_cat_class_0': 0.0681610410285528, 'weight_cat_class_1': 0.6677604097668918, 'weight_cat_class_2': 0.041784694666791335}. Best is trial 1765 with value: 0.9497917773888238.
[I 2026-07-03 10:24:02,191] Trial 1998 finished with value: 0.948502256141365 and parameters: {'weight_lgbm_class_0': 0.09161373121216802, 'weight_lgbm_class_1': 0.7451618114699561, 'weight_lgbm_class_2': 0.8567622297331116, 'weight_xgb_class_0': 0.09420475386401642, 'weight_xgb_class_1': 0.46593431860774287, 'weight_xgb_class_2': 0.6532249588901197, 'weight_cat_class_0': 0.06760819035388299, 'weight_cat_class_1': 0.766827066154781, 'weight_cat_class_2': 0.19827

In [33]:
best_params = study.best_params

w_lgbm = np.array([best_params['weight_lgbm_class_0'], best_params['weight_lgbm_class_1'], best_params['weight_lgbm_class_2']])
w_xgb  = np.array([best_params['weight_xgb_class_0'],  best_params['weight_xgb_class_1'],  best_params['weight_xgb_class_2']])
w_cat  = np.array([best_params['weight_cat_class_0'],  best_params['weight_cat_class_1'],  best_params['weight_cat_class_2']])

proba_lgbm_test = X_test[['lgbm_0', 'lgbm_1', 'lgbm_2']].to_numpy()
proba_xgb_test  = X_test[['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
proba_cat_test  = X_test[['cat_0', 'cat_1', 'cat_2']].to_numpy()

blend_proba_test = (proba_lgbm_test * w_lgbm) + (proba_xgb_test * w_xgb) + (proba_cat_test * w_cat)

pred = np.argmax(blend_proba_test, axis=1)

In [38]:
w_lgbm

array([0.09605921, 0.77203945, 0.99926204])

In [39]:
w_xgb

array([0.00083109, 0.43901447, 0.44084632])

In [40]:
w_cat

array([0.02535431, 0.76277886, 0.11526142])

In [34]:
sub_labels = label_encoder.inverse_transform(pred)

In [35]:
np.unique(pred)

array([0, 1, 2])

# Submission

In [36]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.3.0_xgboost_lightgbm_catboost_submission.csv', index=False)

In [37]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
